# PatchCore DINOv2 ViT-B/14 (`x224`)

This notebook is the canonical evaluation workflow for PatchCore built on a frozen DINOv2 ViT-B/14 backbone.

DINOv2 uses the same ViT-B architecture as the supervised baseline but is pretrained with DINO self-distillation on 142M curated images. Its patch tokens retain strong local spatial structure throughout all layers, unlike supervised ViT which collapses to class-discriminative representations at deeper blocks. This makes it a natural backbone for patch-level anomaly detection.

**No training is performed.** The backbone is fully frozen. The only compute is one forward pass over the training set to build the memory bank.

**Control flags (set in the Configuration cell):**
- `FORCE_REBUILD_SCORES = False` - loads saved scores from `scores.npz`. Set `True` to rebuild the memory bank and re-score.
- `FORCE_RERUN_UMAP = False` - displays the saved UMAP figure. Set `True` to reproject.

Artifacts written by this notebook:
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/checkpoints`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/checkpoints)
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/plots`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/plots)
- [`experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/results`](experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts/results)

## Comparison baseline

| Backbone | Patch size | Tokens/image | Pretraining | F1 | AUROC | AUPRC |
|---|---|---|---|---|---|---|
| ViT-B/16 augreg (frozen) | 16 | 196 | Supervised ImageNet-21k | 0.595 | 0.956 | 0.671 |
| ViT-B/14 DINOv2 (frozen) | 14 | 256 | DINO self-distillation, 142M images | TBD | TBD | TBD |

## Setup

This cell installs or checks optional notebook dependencies before the rest of the workflow runs.

In [ ]:
# -- 0. Install dependencies if needed -----------------------------------------
import importlib.util
import subprocess
import sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')

## Imports

In [ ]:
# -- 1. Imports ----------------------------------------------------------------
import os, gc, json, random, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')

from IPython.display import display, Image as IPImage

## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.

**DINOv2 note:** patch size 14 gives 256 tokens per image at 224x224 (vs 196 for ViT-B/16). All other PatchCore settings are held identical to the frozen ViT-B/16 baseline for a controlled comparison.

In [ ]:
# -- 2. Configuration ----------------------------------------------------------
from pathlib import Path

# -- Repo root resolution ------------------------------------------------------
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

# -- Control flags -------------------------------------------------------------
FORCE_REBUILD_SCORES = False   # True -> rebuild memory bank and re-score all splits
FORCE_RERUN_UMAP     = False   # True -> rerun UMAP projection

# -- Data ----------------------------------------------------------------------
DATA_PATH  = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE = 224

TRAIN_NORMAL_N = 40_000
TUNE_NORMAL_N  =  5_000
TEST_NORMAL_N  =  5_000
TEST_DEFECT_N  =    250

# -- DINOv2 backbone -----------------------------------------------------------
# ViT-B/14 pretrained with DINO self-distillation on LVD-142M.
# patch_size=14 -> 224/14 = 16 -> 16x16 = 256 tokens per image.
# Block 9 is used: DINOv2 retains strong local structure at deeper blocks
# unlike supervised ViT which collapses spatially at block 11.
BACKBONE_NAME     = 'vit_base_patch14_dinov2.lvd142m'
VIT_FEATURE_BLOCK = 9      # sweep candidate: try 6, 9, 11
PATCH_EMBED_DIM   = 128    # same as frozen ViT-B/16 baseline

# -- PatchCore -----------------------------------------------------------------
# All settings identical to frozen ViT-B/16 baseline for controlled comparison.
MEMORY_BANK_MAX  = 600_000
SCORE_CHUNK      = 512
PATCHCORE_NN_K   = 3
TOPK_PATCH_RATIO = 0.1
BATCH_SIZE       = 128
NUM_WORKERS      = 0

# -- Threshold -----------------------------------------------------------------
THRESHOLD_QUANTILE = 0.95

# -- Artifact paths ------------------------------------------------------------
ARTIFACT_DIR    = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/dinov2_vit_b14/x224/main/artifacts')
CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
PLOTS_DIR       = os.path.join(ARTIFACT_DIR, 'plots')
RESULTS_DIR     = os.path.join(ARTIFACT_DIR, 'results')

MODEL_EXPORT_PATH   = os.path.join(CHECKPOINTS_DIR, 'patchcore_dinov2_model.pt')
SCORES_EXPORT_PATH  = os.path.join(RESULTS_DIR, 'scores.npz')
METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
SUMMARY_EXPORT_PATH = os.path.join(RESULTS_DIR, 'summary.json')
UMAP_PNG_PATH       = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
UMAP_CSV_PATH       = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')

for d in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Backbone     : {BACKBONE_NAME}')
print(f'Feature block: {VIT_FEATURE_BLOCK}  embed_dim={PATCH_EMBED_DIM}')
print(f'Bank cap     : {MEMORY_BANK_MAX:,}  topk_ratio={TOPK_PATCH_RATIO}  k={PATCHCORE_NN_K}')
print(f'Threshold    : q={THRESHOLD_QUANTILE}')
print(f'FORCE_REBUILD_SCORES={FORCE_REBUILD_SCORES}  FORCE_RERUN_UMAP={FORCE_RERUN_UMAP}')

## Load Dataset

In [ ]:
# -- 3. Load & clean dataset ---------------------------------------------------
df = pd.read_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None: return 'unknown'
    if isinstance(v, float) and np.isnan(v): return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)

df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()

invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)

normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()

print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print('\nDefect breakdown:')
print(defect_df['failure_label'].value_counts())

## Split

In [ ]:
# -- 4. Split ------------------------------------------------------------------
rng = np.random.default_rng(SEED)
ns  = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds  = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)

a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N

train_normal_df = ns.iloc[0:a].copy()
tune_normal_df  = ns.iloc[a:b].copy()
test_normal_df  = ns.iloc[b:c].copy()
test_defect_df  = ds.iloc[0:TEST_DEFECT_N].copy()

del df, normal_df, defect_df, ns, ds
gc.collect()

print(f'Train normal : {len(train_normal_df):>7,}  (memory bank)')
print(f'Tune  normal : {len(tune_normal_df):>7,}  (threshold calibration)')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')
print('\nDefect classes in test set:')
print(test_defect_df['failure_label'].value_counts())

## Dataset

Lazy dataset: raw numpy wafer maps stored in the DataFrame, converted to tensors per batch only.

In [ ]:
# -- 5. Lazy WaferDataset ------------------------------------------------------
class WaferDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, size: int = 224):
        self.maps   = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size   = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x   = torch.tensor(arr, dtype=torch.long)
        x   = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()  # [3, H, W]
        x   = F.interpolate(
                  x.unsqueeze(0),
                  size=(self.size, self.size),
                  mode='nearest',
              ).squeeze(0)
        return x, int(self.labels[idx])

loader_kw = dict(
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = USE_CUDA,
    persistent_workers = (NUM_WORKERS > 0),
)

train_loader       = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
tune_normal_loader = DataLoader(WaferDataset(tune_normal_df,  IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df,  IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df,  IMAGE_SIZE), **loader_kw)

# Smoke-test
xb, yb = next(iter(train_loader))
print(f'Batch shape: {tuple(xb.shape)}  dtype={xb.dtype}')

## Define Model

DINOv2 ViT-B/14 loaded with pretrained weights. Fully frozen. A forward hook captures block `VIT_FEATURE_BLOCK` patch tokens, which are projected to `PATCH_EMBED_DIM` dimensions for the memory bank.

**Why block 9?** DINOv2's self-distillation training preserves local spatial structure through deeper layers than supervised ViT. Block 9 retains strong patch-level features while incorporating more semantic context than block 6. Block 11 (final) can also be tried - unlike supervised ViT where it collapses, DINOv2 block 11 still retains useful local geometry.

In [ ]:
# -- 6. DINOv2 patch extractor -------------------------------------------------
class DINOv2PatchExtractor(nn.Module):
    """
    Frozen DINOv2 ViT-B/14 backbone.
    Hooks block[VIT_FEATURE_BLOCK] to capture [B, N+1, 768] token output.
    Drops CLS token -> [B, 256, 768] spatial patch tokens.
    Projects to [B, 256, PATCH_EMBED_DIM].
    """

    def __init__(self, block_idx: int = VIT_FEATURE_BLOCK,
                 proj_dim:  int = PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model(
            BACKBONE_NAME,
            pretrained=True,
            num_classes=0,
        )
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(
            lambda m, i, o: setattr(self, '_feat', o)
        )
        embed_dim = self.vit.embed_dim  # 768 for ViT-B
        self.proj = nn.Linear(embed_dim, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)
        # DINOv2 prepends CLS token: output is [B, 257, 768]
        # Drop CLS -> [B, 256, 768] spatial patch tokens
        return self._feat[:, 1:, :]

extractor = DINOv2PatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False

# Smoke-test
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out   = extractor(dummy)
    proj  = extractor.proj(out)

n_tokens = out.shape[1]   # should be 256 for patch_size=14 at 224x224
print(f'Backbone     : {BACKBONE_NAME}')
print(f'Block-{VIT_FEATURE_BLOCK} output : {tuple(out.shape)}  ({n_tokens} tokens/image)')
print(f'After proj   : {tuple(proj.shape)}')
print(f'Tokens vs ViT-B/16: {n_tokens} vs 196 ({n_tokens-196:+d})')

## Train and Score

Builds the PatchCore memory bank (one forward pass over training normals) and scores all splits. When `FORCE_REBUILD_SCORES=False` and `scores.npz` exists, loads saved scores and skips the memory bank build entirely.

In [ ]:
# -- 7. Memory bank or load saved scores ---------------------------------------
REQUIRED_KEYS = {'tune_normal_scores', 'test_normal_scores', 'test_defect_scores', 'threshold'}

saved_keys = set()
if os.path.exists(SCORES_EXPORT_PATH):
    with np.load(SCORES_EXPORT_PATH) as _f:
        saved_keys = set(_f.files)

REUSE_SCORES = REQUIRED_KEYS.issubset(saved_keys) and not FORCE_REBUILD_SCORES
memory_bank  = None

if REUSE_SCORES:
    with np.load(SCORES_EXPORT_PATH) as d:
        tune_normal_scores = d['tune_normal_scores']
        test_normal_scores = d['test_normal_scores']
        test_defect_scores = d['test_defect_scores']
        best_thresh        = float(d['threshold'])
    print(f'Loaded saved scores from: {SCORES_EXPORT_PATH}')
    print(f'Threshold: {best_thresh:.6f}')

else:
    # -- Embedding helper ------------------------------------------------------
    def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
        """L2-normalised patch embeddings: [B*N_tokens, proj_dim] on GPU."""
        with torch.inference_mode():
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)          # [B, 256, 768]
                emb  = extractor.proj(feat)   # [B, 256, proj_dim]
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)
        return emb

    # -- Build memory bank -----------------------------------------------------
    sampled, total_seen, sample_ratio = [], 0, None
    print('Building memory bank...')
    bank_iter = tqdm(enumerate(train_loader), total=len(train_loader),
                     desc='Bank build', unit='batch')
    for step, (xb, _) in bank_iter:
        xb  = xb.to(DEVICE)
        emb = extract_embeddings(xb)
        total_seen += len(emb)

        if sample_ratio is None:
            tokens_per_img  = len(emb) // len(xb)
            estimated_total = tokens_per_img * len(train_normal_df)
            sample_ratio    = min(1.0, MEMORY_BANK_MAX / estimated_total)
            print(f'  Tokens/image : {tokens_per_img}  (DINOv2 patch14 at {IMAGE_SIZE}px)')
            print(f'  Est. total   : {estimated_total:,}')
            print(f'  Sample ratio : {sample_ratio:.5f}')

        if sample_ratio < 1.0:
            k   = max(1, int(round(len(emb) * sample_ratio)))
            idx = torch.randperm(len(emb), device=DEVICE)[:k]
            emb = emb[idx]

        sampled.append(emb)
        if (step + 1) % 20 == 0 or (step + 1) == len(train_loader):
            bank_iter.set_postfix(bank_tokens=f'{sum(len(e) for e in sampled):,}')

    memory_bank = torch.cat(sampled, dim=0)
    del sampled; gc.collect()

    if len(memory_bank) > MEMORY_BANK_MAX:
        idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX]
        memory_bank = memory_bank[idx]

    vram_mb = memory_bank.element_size() * memory_bank.nelement() / 1e6
    print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({vram_mb:.1f} MB VRAM)')

    # -- Score function --------------------------------------------------------
    @torch.inference_mode()
    def score_loader(loader: DataLoader, desc: str = 'Scoring') -> np.ndarray:
        scores = []
        for xb, _ in tqdm(loader, desc=desc, leave=False):
            xb  = xb.to(DEVICE)
            emb = extract_embeddings(xb)
            B   = len(xb)
            P   = emb.shape[0] // B

            patch_dists = torch.empty(B * P, dtype=torch.float32, device=DEVICE)
            for start in range(0, B * P, SCORE_CHUNK):
                end   = min(start + SCORE_CHUNK, B * P)
                chunk = emb[start:end]
                dists = torch.cdist(chunk, memory_bank)
                knn   = dists.topk(PATCHCORE_NN_K, dim=1, largest=False).values
                patch_dists[start:end] = knn.mean(dim=1)

            patch_dists = patch_dists.view(B, P)
            topk        = max(1, int(P * TOPK_PATCH_RATIO))
            img_scores  = patch_dists.topk(topk, dim=1).values.mean(dim=1)
            scores.extend(img_scores.cpu().tolist())
        return np.array(scores)

    print('Scoring splits...')
    tune_normal_scores = score_loader(tune_normal_loader, 'Score tune-normal')
    test_normal_scores = score_loader(test_normal_loader, 'Score test-normal')
    test_defect_scores = score_loader(test_defect_loader, 'Score test-defect')

    best_thresh = float(np.quantile(tune_normal_scores, THRESHOLD_QUANTILE))
    print(f'Threshold (q={THRESHOLD_QUANTILE:.2f} on tune-normal): {best_thresh:.6f}')

    np.savez_compressed(
        SCORES_EXPORT_PATH,
        tune_normal_scores=tune_normal_scores,
        test_normal_scores=test_normal_scores,
        test_defect_scores=test_defect_scores,
        threshold=np.array(best_thresh),
    )
    print(f'Scores saved -> {SCORES_EXPORT_PATH}')

print(f'tune_normal: mean={tune_normal_scores.mean():.4f}  std={tune_normal_scores.std():.4f}')

## Threshold Calibration

95th percentile of tune-normal scores. No defect labels used.

In [ ]:
# -- 8. Threshold report -------------------------------------------------------
best_fpr = float((tune_normal_scores > best_thresh).mean())
print(f'Threshold : {best_thresh:.6f}  (q={THRESHOLD_QUANTILE:.2f})')
print(f'FPR on tune-normal: {best_fpr:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tune_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Tune normal')
ax.axvline(best_thresh, color='red', linestyle='--', lw=2,
           label=f'Threshold={best_thresh:.4f} (q={THRESHOLD_QUANTILE:.2f})')
ax.set_title('Tune-Normal Score Distribution (DINOv2 ViT-B/14)')
ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Count')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'threshold_selection.png'), dpi=140, bbox_inches='tight')
plt.show()

## Evaluate

Overall and per-class metrics at the calibrated threshold.

In [ ]:
# -- 9. Final evaluation -------------------------------------------------------
all_scores = np.concatenate([test_normal_scores, test_defect_scores])
all_labels = np.concatenate([
    np.zeros(len(test_normal_scores), dtype=int),
    np.ones( len(test_defect_scores), dtype=int),
])
predictions = (all_scores > best_thresh).astype(int)

roc_auc = roc_auc_score(all_labels, all_scores)
auprc   = average_precision_score(all_labels, all_scores)
fpr_arr, tpr_arr, _       = roc_curve(all_labels, all_scores)
precision_arr, recall_arr, _ = precision_recall_curve(all_labels, all_scores)

f1_val        = f1_score(all_labels, predictions, pos_label=1, zero_division=0)
precision_val = precision_score(all_labels, predictions, pos_label=1, zero_division=0)
recall_val    = recall_score(all_labels, predictions, pos_label=1, zero_division=0)

print('-- Overall Test Results --------------------------')
print(f'ROC-AUC  : {roc_auc:.4f}')
print(f'AUPRC    : {auprc:.4f}')
print(f'Threshold: {best_thresh:.6f}  (q={THRESHOLD_QUANTILE:.2f})')
print()
print(classification_report(all_labels, predictions,
                             target_names=['Normal', 'Defect'], digits=4))

# -- Per-class breakdown -------------------------------------------------------
defect_class_labels = test_defect_df['failure_label'].values
defect_preds        = (test_defect_scores > best_thresh).astype(int)

print('-- Per-class defect recall ---------------------------------------')
print(f'  {"Defect type":<14}  {"N":>5}  {"Detected":>8}  {"Recall":>7}  {"Mean score":>10}')
print('  ' + '-' * 52)

perclass_results = {}
for cls in sorted(np.unique(defect_class_labels)):
    mask       = defect_class_labels == cls
    n          = mask.sum()
    detected   = defect_preds[mask].sum()
    recall     = detected / n
    mean_score = test_defect_scores[mask].mean()
    perclass_results[cls] = {
        'n': int(n), 'detected': int(detected),
        'recall': float(recall), 'mean_score': float(mean_score),
    }
    print(f'  {cls:<14}  {n:>5}  {detected:>8}  {recall:>6.1%}  {mean_score:>10.4f}')

overall_defect_recall = defect_preds.sum() / len(defect_preds)
print('  ' + '-' * 52)
print(f'  {"ALL DEFECTS":<14}  {len(defect_preds):>5}  {defect_preds.sum():>8}  {overall_defect_recall:>6.1%}')

# -- Comparison with frozen ViT-B/16 baseline ----------------------------------
print('\n-- vs Frozen ViT-B/16 baseline -------------------')
baseline = {'F1': 0.595, 'AUROC': 0.956, 'AUPRC': 0.671}
print(f'  F1    : {f1_val:.4f}  (baseline {baseline["F1"]:.3f}, delta {f1_val-baseline["F1"]:+.3f})')
print(f'  AUROC : {roc_auc:.4f}  (baseline {baseline["AUROC"]:.3f}, delta {roc_auc-baseline["AUROC"]:+.3f})')
print(f'  AUPRC : {auprc:.4f}  (baseline {baseline["AUPRC"]:.3f}, delta {auprc-baseline["AUPRC"]:+.3f})')

In [ ]:
# -- 10. Evaluation plots ------------------------------------------------------
fig = plt.figure(figsize=(21, 10))
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

ax_dist = fig.add_subplot(gs[0, 0])
ax_roc  = fig.add_subplot(gs[0, 1])
ax_pr   = fig.add_subplot(gs[0, 2])
ax_cm   = fig.add_subplot(gs[0, 3])
ax_pc   = fig.add_subplot(gs[1, :])

# Score distributions
ax_dist.hist(test_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Normal')
ax_dist.hist(test_defect_scores, bins=50, alpha=0.7, color='tomato',    label='Defect')
ax_dist.axvline(best_thresh, color='black', linestyle='--',
                label=f'Threshold={best_thresh:.4f}')
ax_dist.set_title('Score Distributions'); ax_dist.set_xlabel('Anomaly Score')
ax_dist.set_ylabel('Count'); ax_dist.legend(fontsize=9)

# ROC
ax_roc.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'ROC-AUC={roc_auc:.4f}')
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)
op_fpr = (test_normal_scores > best_thresh).mean()
op_tpr = (test_defect_scores > best_thresh).mean()
ax_roc.scatter([op_fpr], [op_tpr], color='red', zorder=5,
               label=f'FPR={op_fpr:.3f} TPR={op_tpr:.3f}')
ax_roc.set_title('ROC Curve')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR'); ax_roc.legend(fontsize=9)

# PR
baseline_pr = all_labels.mean()
ax_pr.plot(recall_arr, precision_arr, color='purple', lw=2, label=f'AUPRC={auprc:.4f}')
ax_pr.axhline(baseline_pr, color='gray', linestyle='--', lw=1,
              label=f'No-skill={baseline_pr:.3f}')
ax_pr.set_title('Precision-Recall'); ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_xlim(0, 1); ax_pr.set_ylim(0, 1.02); ax_pr.legend(fontsize=9)

# Confusion matrix
cm = confusion_matrix(all_labels, predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal', 'Pred Defect'],
            yticklabels=['True Normal', 'True Defect'], ax=ax_cm)
ax_cm.set_title('Confusion Matrix')

# Per-class recall
classes     = list(perclass_results.keys())
recalls     = [perclass_results[c]['recall']     for c in classes]
counts      = [perclass_results[c]['n']          for c in classes]
mean_scores = [perclass_results[c]['mean_score'] for c in classes]

order   = np.argsort(recalls)
classes = [classes[i] for i in order]
recalls = [recalls[i] for i in order]
counts  = [counts[i]  for i in order]
mean_scores = [mean_scores[i] for i in order]

bar_colors = ['#e05c5c' if r < 0.5 else '#f5a623' if r < 0.8 else '#4caf50' for r in recalls]
bars = ax_pc.bar(classes, recalls, color=bar_colors, edgecolor='white', width=0.6)
ax_pc.axhline(overall_defect_recall, color='navy', linestyle='--', lw=1.5,
              label=f'Overall recall={overall_defect_recall:.1%}')
ax_pc.axhline(0.8, color='gray', linestyle=':', lw=1, label='80% target')
ax_pc.set_ylim(0, 1.12); ax_pc.set_ylabel('Detection Recall')
ax_pc.set_title('Per-Class Defect Recall  (red<50% | orange 50-80% | green>=80%)')
ax_pc.legend(fontsize=9)

for bar, rec, n in zip(bars, recalls, counts):
    ax_pc.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{rec:.0%}\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle(
    f'PatchCore DINOv2 ViT-B/14 block-{VIT_FEATURE_BLOCK}  |  '
    f'ROC-AUC={roc_auc:.4f}  |  AUPRC={auprc:.4f}  |  F1={f1_val:.4f}',
    fontsize=13, fontweight='bold',
)
plt.savefig(os.path.join(PLOTS_DIR, 'evaluation_results.png'), dpi=130, bbox_inches='tight')
plt.show()

defect_recall_df = pd.DataFrame([
    {'failure_label': c, **v} for c, v in perclass_results.items()
]).sort_values('recall').reset_index(drop=True)
defect_recall_df.to_csv(os.path.join(RESULTS_DIR, 'defect_recall.csv'), index=False)
display(defect_recall_df)

## UMAP Visualization

Projects mean-pooled patch embeddings into 2D. Displays saved PNG when `FORCE_RERUN_UMAP=False`.

In [ ]:
# -- 11. UMAP ------------------------------------------------------------------
from sklearn.decomposition import PCA

if (not FORCE_RERUN_UMAP) and os.path.exists(UMAP_PNG_PATH):
    print(f'Displaying saved UMAP: {UMAP_PNG_PATH}')
    display(IPImage(filename=UMAP_PNG_PATH))
else:
    try:
        import umap.umap_ as umap
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        import umap.umap_ as umap

    MAX_UMAP = 2000

    def collect_image_embeddings(loader, max_images=2000, desc='embed'):
        embs, labels = [], []
        seen = 0
        with torch.inference_mode():
            for xb, yb in tqdm(loader, desc=desc, unit='batch'):
                xb = xb.to(DEVICE)
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb  = extractor.proj(feat)
                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                embs.append(img_emb.cpu().numpy())
                labels.append(yb.cpu().numpy())
                seen += len(yb)
                if seen >= max_images:
                    break
        return np.concatenate(embs)[:max_images], np.concatenate(labels)[:max_images]

    print('Collecting embeddings for UMAP...')
    Xn, yn = collect_image_embeddings(test_normal_loader, MAX_UMAP, 'Embed test-normal')
    Xd, yd = collect_image_embeddings(test_defect_loader, MAX_UMAP, 'Embed test-defect')

    X = np.concatenate([Xn, Xd])
    y = np.concatenate([yn, yd]).astype(int)

    n_pca = min(32, X.shape[1], X.shape[0] - 1)
    Xp = PCA(n_components=n_pca, random_state=SEED).fit_transform(X)

    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=SEED, transform_seed=SEED,
                        low_memory=True, verbose=True)
    coords = reducer.fit_transform(Xp)

    fig, ax = plt.subplots(figsize=(8, 6))
    m0, m1 = (y == 0), (y == 1)
    ax.scatter(coords[m0, 0], coords[m0, 1], s=8, alpha=0.45, c='steelblue', label='Normal', linewidths=0)
    ax.scatter(coords[m1, 0], coords[m1, 1], s=8, alpha=0.60, c='tomato',    label='Defect', linewidths=0)
    ax.set_title(f'UMAP - DINOv2 ViT-B/14 PatchCore (block {VIT_FEATURE_BLOCK})')
    ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2'); ax.legend()
    plt.tight_layout()
    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
    plt.show()

    pd.DataFrame({'umap_1': coords[:, 0], 'umap_2': coords[:, 1], 'label': y}).to_csv(UMAP_CSV_PATH, index=False)
    print(f'Saved: {UMAP_PNG_PATH}')

## Save Outputs

In [ ]:
# -- 12. Save artifacts --------------------------------------------------------
if memory_bank is not None:
    torch.save({
        'extractor_state': extractor.state_dict(),
        'memory_bank':     memory_bank.cpu(),
        'threshold':       float(best_thresh),
        'threshold_quantile': float(THRESHOLD_QUANTILE),
        'config': dict(
            backbone=BACKBONE_NAME,
            vit_feature_block=VIT_FEATURE_BLOCK,
            patch_embed_dim=PATCH_EMBED_DIM,
            image_size=IMAGE_SIZE,
            train_normal_n=TRAIN_NORMAL_N,
            tune_normal_n=TUNE_NORMAL_N,
            test_normal_n=TEST_NORMAL_N,
            test_defect_n=TEST_DEFECT_N,
            memory_bank_max=MEMORY_BANK_MAX,
            patchcore_nn_k=PATCHCORE_NN_K,
            topk_patch_ratio=TOPK_PATCH_RATIO,
            threshold_quantile=THRESHOLD_QUANTILE,
        ),
    }, MODEL_EXPORT_PATH)
    print(f'Model saved -> {MODEL_EXPORT_PATH}')
else:
    print('Scores reused from disk - model artifact unchanged.')

metrics = dict(
    roc_auc          = float(roc_auc),
    auprc            = float(auprc),
    f1_defect        = float(f1_val),
    precision_defect = float(precision_val),
    recall_defect    = float(recall_val),
    threshold        = float(best_thresh),
    threshold_quantile = float(THRESHOLD_QUANTILE),
    confusion_matrix = cm.tolist(),
    n_test_normal    = int(len(test_normal_scores)),
    n_test_defect    = int(len(test_defect_scores)),
    per_class        = perclass_results,
)
with open(METRICS_EXPORT_PATH, 'w') as fh:
    json.dump(metrics, fh, indent=2)

summary = dict(
    name       = f'PatchCore DINOv2 ViT-B/14 block-{VIT_FEATURE_BLOCK}',
    checkpoint_path = MODEL_EXPORT_PATH,
    metrics    = metrics,
    config     = dict(
        backbone=BACKBONE_NAME,
        vit_feature_block=VIT_FEATURE_BLOCK,
        patch_embed_dim=PATCH_EMBED_DIM,
        memory_bank_max=MEMORY_BANK_MAX,
        patchcore_nn_k=PATCHCORE_NN_K,
        topk_patch_ratio=TOPK_PATCH_RATIO,
        threshold_quantile=THRESHOLD_QUANTILE,
    ),
)
with open(SUMMARY_EXPORT_PATH, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f'Metrics -> {METRICS_EXPORT_PATH}')
print(f'Summary -> {SUMMARY_EXPORT_PATH}')
print(f'\nROC-AUC : {roc_auc:.4f}')
print(f'AUPRC   : {auprc:.4f}')
print(f'F1      : {f1_val:.4f}')

## Cleanup

In [ ]:
# -- 13. Cleanup ---------------------------------------------------------------
for name in [
    'extractor', 'memory_bank',
    'tune_normal_scores', 'test_normal_scores', 'test_defect_scores',
    'all_scores', 'all_labels', 'predictions',
    'train_loader', 'tune_normal_loader', 'test_normal_loader', 'test_defect_loader',
]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Cleared.')